# 03 · Evaluation metrics

All scores are computed in sentence space against the creator's chapters and macro-averaged over **every requested
video**. A video whose output fails strict parsing stays in the average with the worst score: 0 for F1/similarity, 1 for Pk/WindowDiff.

| Metric | Meaning |
|---|---|
| **Exact F1** (primary) | Boundary = first sentence of every chapter after the first. F1 counts only exact index matches. |
| ±2 F1 | Same, but a predicted boundary within 2 sentences of an unmatched reference boundary also counts (one-to-one). |
| Pk ↓ | Probe of k sentences (half the mean reference chapter length). Error when reference and prediction disagree on whether both ends are in one chapter. |
| WindowDiff ↓ | Same probe; error when the *number* of boundaries inside it differs. |
| Boundary Similarity | segeval edit-distance similarity; near misses get partial credit. |
| Title ROUGE-L | ROUGE-L F1 between the concatenated predicted and reference titles of each video. |
| Title BERTScore | Chapters paired by span (IoU ≥ 0.5, titles never read for pairing); RoBERTa-large layer-17 F1 per pair; unmatched reference chapters count 0. |
| Paired bootstrap | Per-video exact-F1 difference between two systems; 10,000 video resamples; 95% percentile interval. |

In [1]:
%matplotlib inline
import json, statistics
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import pandas as pd
from nbcommon import *

pd.set_option("display.max_colwidth", 80)
RUN_GPU = False   # True only on the training/inference machine (needs the models and a GPU)
print("project:", ROOT.name, "| run outputs read from:", A.relative_to(ROOT))

project: FinalProject | run outputs read from: artifacts


## 1. A toy example

In [2]:
from chaptering.evaluation import score_prediction
reference = {"video_id": "toy", "sentences": [f"s{i}" for i in range(20)],
             "reference": {"chapter_starts": [0, 5, 12], "chapter_titles": ["Intro", "Setup", "Results"]}}
prediction = {"video_id": "toy", "status": "ok",
              "chapters": [{"start": 0, "title": "Intro"}, {"start": 6, "title": "Setting up"}, {"start": 12, "title": "Results"}]}
s = score_prediction(reference, prediction)
print("exact   :", {k: s["exact"][k] for k in ["tp", "fp", "fn", "f1"]})
print("±2      :", {k: s["tolerant"][k] for k in ["tp", "fp", "fn", "f1"]})
print("Pk / WD :", s["structural"])
failed = score_prediction(reference, {"video_id": "toy", "status": "invalid", "chapters": []})
print("failed  :", failed["exact"]["f1"], failed["structural"]["pk"])

exact   : {'tp': None, 'fp': None, 'fn': None, 'f1': 0.0}
±2      : {'tp': None, 'fp': None, 'fn': None, 'f1': 0.0}
Pk / WD : {'pk': 1.0, 'window_diff': 1.0, 'window_size': None, 'positions': None}
failed  : 0.0 1.0


## 2. Re-scoring a saved run and checking it against the saved evaluation

In [3]:
# E2 LoRA on the 200 dev videos: re-score the saved predictions and compare with the saved paired evaluation file.
ft = recompute(DEV_RECORDS, BACKBONES["E2"]["runs"][("dev", "FT")] / "predictions.jsonl")
zs = recompute(DEV_RECORDS, BACKBONES["E2"]["runs"][("dev", "ZS")] / "predictions.jsonl")
saved_ft, saved_zs, saved_boot = paired_metrics(paired_file("E2", "dev"))
print("recomputed exact F1  LoRA %.4f  zero-shot %.4f" % (ft["aggregate"]["macro_exact_f1"], zs["aggregate"]["macro_exact_f1"]))
print("saved      exact F1  LoRA %.4f  zero-shot %.4f" % (saved_ft["exact_f1"], saved_zs["exact_f1"]))
assert abs(ft["aggregate"]["macro_exact_f1"] - saved_ft["exact_f1"]) < 1e-12

recomputed exact F1  LoRA 0.5120  zero-shot 0.1911
saved      exact F1  LoRA 0.5120  zero-shot 0.1911


In [4]:
boot = bootstrap(ft, zs)
print("paired bootstrap (LoRA − zero-shot): %+.4f  95%% CI [%+.4f, %+.4f]" % (boot["mean_difference"], *boot["confidence_interval"]))
print("saved:                               %+.4f  95%% CI [%+.4f, %+.4f]" % (saved_boot["mean_difference"], *saved_boot["confidence_interval"]))

paired bootstrap (LoRA − zero-shot): +0.3209  95% CI [+0.2761, +0.3665]
saved:                               +0.3209  95% CI [+0.2761, +0.3665]


## 3. Full evaluation command

The title metrics use `rouge-score`, `segeval` and `bert-score`, with RoBERTa-large pinned to a commit. All of them run in `scripts/evaluate_predictions.py`, which writes the paired files used throughout the report:

In [5]:
run(evaluate_command(DEV_RECORDS, BACKBONES["E2"]["runs"][("dev", "FT")] / "predictions.jsonl", S5 / "E2-dev-paired.json",
                     BACKBONES["E2"]["runs"][("dev", "ZS")] / "predictions.jsonl"), RUN_GPU)
display(metric_table([("E2 zero-shot (dev)", saved_zs), ("E2 LoRA (dev)", saved_ft)]))

[not run here; RUN_GPU=False]  $ python scripts/evaluate_predictions.py --references artifacts/windows/dev-v2/validation_records.jsonl --predictions artifacts/stage5-v1/E2-dev-FT/predictions.jsonl --compare-predictions artifacts/stage5-v1/E2-dev-ZS/predictions.jsonl --output artifacts/stage5-v1/E2-dev-paired.json --semantic-model FacebookAI/roberta-large --semantic-revision 722cf37b1afa9454edce342e7895e588b6ff1d59 --semantic-layers 17 --semantic-device cpu


,videos,failed,exact_f1,tolerant_f1,pk,window_diff,boundary_similarity,title_rouge_l,title_bertscore,pred_per_ref
configuration,,,,,,,,,,
E2 zero-shot (dev),200,67,0.191,0.293,0.561,0.582,0.176,0.183,0.241,0.585
E2 LoRA (dev),200,1,0.512,0.680,0.225,0.245,0.486,0.375,0.644,0.945
